In [1]:
import pandas as pd

# Load data
df = pd.read_csv("../data/synthetic_er_data.csv")

# Quick look
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   patient_id              1000 non-null   int64 
 1   age                     1000 non-null   int64 
 2   gender                  1000 non-null   object
 3   arrival_time            1000 non-null   object
 4   triage_score            1000 non-null   int64 
 5   heart_rate              1000 non-null   int64 
 6   systolic_bp             1000 non-null   int64 
 7   diastolic_bp            1000 non-null   int64 
 8   arrival_to_triage_mins  1000 non-null   int64 
 9   visit_reason            1000 non-null   object
 10  bp_delta                1000 non-null   int64 
 11  admitted                1000 non-null   int64 
 12  readmitted              1000 non-null   int64 
dtypes: int64(10), object(3)
memory usage: 101.7+ KB


,patient_id,age,gender,arrival_time,triage_score,heart_rate,systolic_bp,diastolic_bp,arrival_to_triage_mins,visit_reason,bp_delta,admitted,readmitted
0,1,69,Male,2024-05-01 01:29:00,2,89,126,78,29,Shortness of Breath,48,0,1
1,2,32,Male,2024-05-01 01:26:00,4,98,121,78,3,Injury,43,1,1
2,3,89,Female,2024-05-01 01:48:00,4,84,120,76,11,Injury,44,1,1
3,4,78,Male,2024-05-01 00:10:00,2,104,159,93,23,Dizziness,66,1,1
4,5,38,Male,2024-05-01 01:45:00,3,67,149,86,14,Fever,63,0,0


In [2]:
# Cell 2 — Parse & Sanity‐check arrival_time
import pandas as pd
import numpy as np
import os

# 1. Load raw CSV
df = pd.read_csv("../data/synthetic_er_data.csv")

# 2. Convert arrival_time to datetime so we can extract time‐based features
df['arrival_time'] = pd.to_datetime(df['arrival_time'])

# 3. Quick sanity checks
print("Time range:", df['arrival_time'].min(), "–", df['arrival_time'].max())
print("Sample rows:")
df[['patient_id','arrival_time','triage_score','heart_rate']].head(5)


Time range: 2024-05-01 00:00:00 – 2024-05-01 09:43:00
Sample rows:


,patient_id,arrival_time,triage_score,heart_rate
0,1,2024-05-01 01:29:00,2,89
1,2,2024-05-01 01:26:00,4,98
2,3,2024-05-01 01:48:00,4,84
3,4,2024-05-01 00:10:00,2,104
4,5,2024-05-01 01:45:00,3,67


In [3]:
# Cell 3 — Core Feature Engineering
# 1. Blood‐pressure delta: difference between systolic/diastolic
df['bp_delta'] = df['systolic_bp'] - df['diastolic_bp']

# 2. Arrival hour: captures daily cycle effects
df['arrival_hour'] = df['arrival_time'].dt.hour

# 3. High‐HR flag: heart rate above 100 bpm
df['high_hr_flag'] = (df['heart_rate'] > 100).astype(int)

# 4. Urgent triage: triage scores 1 or 2 are high urgency
df['triage_urgent'] = df['triage_score'] <= 2

# 5. Wait time category: short (<10 min), medium (10–30), long (>30)
df['wait_cat'] = pd.cut(
    df['arrival_to_triage_mins'], 
    bins=[0,10,30, np.inf], 
    labels=['short','medium','long']
)


In [4]:
# Cell 4 — One‐Hot Encoding & Visitor Flags
# 1. One‐hot encode visit_reason
reason_dummies = pd.get_dummies(df['visit_reason'], prefix='reason', drop_first=True)
df = pd.concat([df, reason_dummies], axis=1)

# 2. Frequent‐visitor flag: count how many visits per patient
df['visit_count'] = df.groupby('patient_id')['patient_id'].transform('count')
df['frequent_visitor_flag'] = (df['visit_count'] > 1).astype(int)

# 3. Drop or archive columns you won’t feed directly into models
drop_cols = ['visit_reason', 'systolic_bp','diastolic_bp']
df_model = df.drop(columns=drop_cols)


In [5]:
# Cell 5 — Robust Encoding, Split & Save

import os
from sklearn.model_selection import train_test_split

# --- 1) Dynamically encode any categoricals that exist ---
to_encode = [col for col in ['gender','wait_cat'] if col in df_model.columns]
if to_encode:
    df_model = pd.get_dummies(df_model, columns=to_encode, drop_first=True)
    print("Encoded:", to_encode)
else:
    print("No gender/wait_cat columns to encode.")

# --- 2) Collect one-hot prefixes ---
reason_cols = [c for c in df_model.columns if c.startswith('reason_')]
gender_cols = [c for c in df_model.columns if c.startswith('gender_')]
wait_cols   = [c for c in df_model.columns if c.startswith('wait_cat_')]

# --- 3) Base numeric/binary features (only keep ones that exist) ---
base_cols = [
    'age',
    'arrival_hour',
    'bp_delta',
    'high_hr_flag',
    'triage_urgent',
    'visit_count',
    'frequent_visitor_flag'
]
base_cols = [c for c in base_cols if c in df_model.columns]

# --- 4) Final feature list ---
features = base_cols + reason_cols + gender_cols + wait_cols
print("Using features:", features)

# --- 5) Build X, y ---
X = df_model[features]
y = df_model['admitted']  # ensure 'admitted' is in df_model

# --- 6) Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- 7) Save to disk ---
out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

X_train.to_csv(f"{out_dir}/X_train.csv", index=False)
X_test .to_csv(f"{out_dir}/X_test.csv",  index=False)
y_train.to_csv(f"{out_dir}/y_train.csv", index=False)
y_test .to_csv(f"{out_dir}/y_test.csv",  index=False)

print("Train/test splits saved to", out_dir)


Encoded: ['gender', 'wait_cat']
Using features: ['age', 'arrival_hour', 'bp_delta', 'high_hr_flag', 'triage_urgent', 'visit_count', 'frequent_visitor_flag', 'reason_Dizziness', 'reason_Fever', 'reason_Injury', 'reason_Shortness of Breath', 'gender_Male', 'wait_cat_medium', 'wait_cat_long']
Train/test splits saved to ../data/processed
